# Experiment 1: Base Performance (AL)

Instance-level analysis: compile rate, pass rate per model across base entries.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from utils import load_all_results

# 3 CFE runs: claude-opus-4-6, gpt-5-4, gpt-5-3-codex
SETUPS = ["copilot-cf-opus-4-6", "copilot-cf-gpt-5-4", "copilot-cf-gpt-5-3-codex"]
SETUP_LABELS = {
    "copilot-cf-opus-4-6": "Claude Opus 4.6",
    "copilot-cf-gpt-5-4": "GPT-5-4",
    "copilot-cf-gpt-5-3-codex": "GPT-5-3 Codex",
}

all_results = load_all_results(category="bug-fix")
print(f"Loaded {len(all_results)} setups: {list(all_results.keys())}")
for name in SETUPS:
    if name in all_results:
        df = all_results[name]
        print(f"  {name}: {len(df)} results, {df['instance_id'].nunique()} instances")

Loaded 6 setups: ['claude-opus-4-5', 'copilot-cf-gpt-5-3-codex', 'copilot-cf-gpt-5-4', 'copilot-cf-opus-4-6', 'copilot-opus-4-5', 'copilot-opus-4-6']
  copilot-cf-opus-4-6: 626 results, 222 instances
  copilot-cf-gpt-5-4: 121 results, 121 instances
  copilot-cf-gpt-5-3-codex: 121 results, 121 instances


In [2]:
import pandas as pd
import plotly.express as px

summary_rows = []
for setup, df in all_results.items():
    if SETUPS and setup not in SETUPS:
        continue
    label = SETUP_LABELS.get(setup, setup)
    n_runs = df["run_id"].nunique()
    n_instances = df["instance_id"].nunique()
    compile_rate = df["build"].mean() * 100
    pass_rate = df["resolved"].mean() * 100
    summary_rows.append(
        {
            "Model": label,
            "Runs": n_runs,
            "Instances": n_instances,
            "Compile Rate (%)": round(compile_rate, 1),
            "Pass Rate (%)": round(pass_rate, 1),
        }
    )

summary_df = pd.DataFrame(summary_rows)
summary_df

,Model,Runs,Instances,Compile Rate (%),Pass Rate (%)
0,GPT-5-3 Codex,1,121,95.0,60.3
1,GPT-5-4,1,121,93.4,65.3
2,Claude Opus 4.6,6,222,97.1,64.7


In [3]:
# Bar chart: compile rate and pass rate per model
if not summary_df.empty:
    melted = summary_df.melt(
        id_vars=["Model"],
        value_vars=["Compile Rate (%)", "Pass Rate (%)"],
        var_name="Metric",
        value_name="Rate",
    )
    fig = px.bar(
        melted,
        x="Model",
        y="Rate",
        color="Metric",
        barmode="group",
        title="AL Base Performance by Model",
        text_auto=True,
    )
    fig.update_layout(yaxis_range=[0, 100])
    fig.show()

In [4]:
# Per-instance resolution heatmap (instances x models)
if not summary_df.empty:
    frames = []
    for setup, df in all_results.items():
        if SETUPS and setup not in SETUPS:
            continue
        label = SETUP_LABELS.get(setup, setup)
        per_instance = df.groupby("instance_id")["resolved"].mean().reset_index()
        per_instance["Model"] = label
        frames.append(per_instance)

    if frames:
        combined = pd.concat(frames)
        pivot = combined.pivot(index="instance_id", columns="Model", values="resolved")
        fig = px.imshow(
            pivot,
            aspect="auto",
            title="Per-Instance Resolution Rate",
            labels={"color": "Resolved %"},
        )
        fig.show()